# Formation Energies and Convex Hull (Analyzer)

Analyze existing total energy results to compute formation energies and build the convex hull.

<h2 style="color:green">Usage</h2>

1. Set the material set and formulas in section 1 below.
2. Click "Run" > "Run All".
3. The notebook finds materials, retrieves total energies, and builds the convex hull.

**Prerequisites:** Total energy calculations must already be completed for the materials.

## How it works

1. **Find materials** in a material set (or all materials) matching the given formulas
2. **Preview** the materials (formula, structure type, space group)
3. **Retrieve total energies** from completed jobs
4. **Build convex hull** using pymatgen
5. **Analyze stability** — formation energies, energy above hull, decomposition products

## 1. Parameters

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples")

In [ ]:
# Materials to search for.
FORMULAS = ["Hf", "Zr", "O2", "HfO2", "ZrO2"]

# Name (partial match) or ID of a material set to search within.
# Set to None to search all materials in the account.
MATERIAL_SET = None  # e.g., "hf-zr-o-study"

# Organization name (partial match) to search within.
ORGANIZATION_NAME = None

## 2. Authenticate

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()

selected_account = client.my_account
if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)
ACCOUNT_ID = selected_account.id
print(f"✅ Account: {selected_account.name} ({ACCOUNT_ID})")

## 3. Find materials

In [ ]:
# Resolve material set (if specified)
set_id = None
if MATERIAL_SET:
    set_query = {"owner._id": ACCOUNT_ID, "isEntitySet": True,
                 "name": {"$regex": MATERIAL_SET, "$options": "i"}}
    sets = client.materials.list(set_query)
    if not sets:
        raise ValueError(f"No material set matching '{MATERIAL_SET}'")
    set_id = sets[0]["_id"]
    print(f"✅ Using set: {sets[0]['name']} ({set_id})")
else:
    print("ℹ️  No material set specified — searching all materials in account.")

In [ ]:
# Search materials by formula
all_materials = []

for formula in FORMULAS:
    query = {"formula": formula, "owner._id": ACCOUNT_ID}
    if set_id:
        query["inSet._id"] = set_id
    matches = client.materials.list(query)
    # Filter out entity sets
    matches = [m for m in matches if not m.get("isEntitySet")]
    for m in matches:
        m["_search_formula"] = formula  # track which formula query found this
    all_materials.extend(matches)
    print(f"{formula}: {len(matches)} material(s) found")

print(f"\nTotal: {len(all_materials)} materials")

### 3.1. Preview materials

In [ ]:
import pandas as pd

preview = []
for m in all_materials:
    elements = m.get("basis", {}).get("elements", [])
    n_atoms = len(m.get("basis", {}).get("coordinates", []))
    lattice_type = m.get("lattice", {}).get("type", "?")
    preview.append({
        "ID": m["_id"],
        "Name": m.get("name", "?"),
        "Formula": m.get("formula", "?"),
        "Lattice": lattice_type,
        "Atoms": n_atoms,
    })

df_preview = pd.DataFrame(preview)
df_preview

## 4. Retrieve total energies

For each material, find completed jobs and extract total energy properties.

In [ ]:
from collections import Counter
from mat3ra.prode import PropertyName

entries_data = []  # list of {material, composition, total_energy, ...}

for m in all_materials:
    mat_id = m["_id"]
    formula = m.get("formula", "?")
    basis = m.get("basis", {})
    elements = basis.get("elements", [])
    n_atoms = len(basis.get("coordinates", []))

    # Derive composition from actual basis elements
    composition = dict(Counter(el["value"] for el in elements))

    # Find jobs that used this material
    jobs = client.jobs.list({
        "owner._id": ACCOUNT_ID,
        "_material._id": mat_id,
        "status": "finished",
    })

    if not jobs:
        print(f"⚠️  {formula} ({mat_id}): no finished jobs found — skipping")
        continue

    # Get total energy from the most recent finished job
    job = jobs[0]
    props = client.properties.get_for_job(
        job["_id"],
        property_name=PropertyName.scalar.total_energy.value
    )

    if not props or props[0].get("value") is None:
        print(f"⚠️  {formula} ({mat_id}): no total energy in job {job['_id']} — skipping")
        continue

    total_energy = props[0]["value"]
    energy_per_atom = total_energy / n_atoms

    entries_data.append({
        "material_id": mat_id,
        "job_id": job["_id"],
        "formula": formula,
        "composition": composition,
        "n_atoms": n_atoms,
        "total_energy": total_energy,
        "energy_per_atom": energy_per_atom,
        "lattice_type": m.get("lattice", {}).get("type", "?"),
        "name": m.get("name", "?"),
    })
    print(f"✅ {formula} ({m.get('lattice', {}).get('type', '?')}): "
          f"E = {total_energy:.4f} eV, E/atom = {energy_per_atom:.4f} eV")

print(f"\n{len(entries_data)} entries with total energies.")

## 5. Build convex hull

In [ ]:
from pymatgen.core import Composition
from pymatgen.entries.computed_entries import ComputedEntry
from pymatgen.analysis.phase_diagram import PhaseDiagram

entries = []
for data in entries_data:
    comp = Composition(data["composition"])
    entry = ComputedEntry(comp, data["total_energy"])
    entries.append(entry)

pd_diagram = PhaseDiagram(entries)
print(f"✅ Convex hull built with {len(entries)} entries.")

## 6. Results

In [ ]:
results = []
for i, entry in enumerate(entries):
    e_above_hull = pd_diagram.get_e_above_hull(entry)
    decomp, _ = pd_diagram.get_decomp_and_e_above_hull(entry)
    decomp_str = " + ".join([e.composition.reduced_formula for e in decomp])
    data = entries_data[i]
    results.append({
        "Formula": entry.composition.reduced_formula,
        "Lattice": data["lattice_type"],
        "E/atom (eV)": round(entry.energy_per_atom, 4),
        "E_form/atom (eV)": round(pd_diagram.get_form_energy_per_atom(entry), 4),
        "E_above_hull (eV)": round(e_above_hull, 4),
        "Stable": "✅" if e_above_hull < 1e-6 else "❌",
        "Decomposes to": decomp_str if e_above_hull > 1e-6 else "—",
    })

df = pd.DataFrame(results).sort_values("E_above_hull (eV)")
df

## 7. Plot

In [ ]:
from pymatgen.analysis.phase_diagram import PDPlotter

fig = PDPlotter(pd_diagram, show_unstable=0.2).get_plot()
for trace in fig.data:
    if trace.hovertext:
        trace.text = trace.hovertext
        trace.mode = "markers+text"
        trace.textposition = "top center"
fig.show()
